# linalg-solve-batched composite — cx11: every-ray every-triangle solve via repeat-broadcast LHS

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat-broadcast`, `linalg-solve-batched`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "linalg-solve-batched"
DD_ATOM_IDS = ["einops-repeat-broadcast", "linalg-solve-batched"]
DD_SUBTOPICS = ["Einops: Repeat-as-broadcast", "PyTorch: Batched linalg.solve"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Generalize cx10 from one triangle to many: solve every (ray, triangle) pair in one batched call. The hard step is building the `(NR, NT, 3, 3)` LHS without materializing redundant copies.

Atoms compose:
  1. `einops-repeat-broadcast` inserts the t-axis on per-ray data and the r-axis on per-tri data, both as stride-0 views.
  2. `linalg-solve-batched` accepts ANY leading batch shape — `(*, n, n)` works for a `(NR, NT, 3, 3)` LHS just as well as a `(NR, 3, 3)`. The batch dimensions can be plural.

Result: `(NR, NT, 3)` of `(s, u, v)` for every pair. The solve broadcasts the rays across triangles and vice-versa for free because repeat-broadcast already aligned the shapes.

### Composite Exercise — every-ray every-triangle solve via repeat-broadcast LHS

**Atoms exercised together**: `einops-repeat-broadcast`, `linalg-solve-batched`

Implement `cx11_intersect_all_pairs(rays, triangles)` — for every (ray, triangle) pair, solve the 3x3 system and return `(NR, NT, 3)` of `(s, u, v)`.

- `rays`:      `(NR, 2, 3)`  — `[origin, direction]` per ray.
- `triangles`: `(NT, 3, 3)` — three vertices A/B/C per triangle.

Steps:
1. Split rays into `O: (NR, 3)`, `D: (NR, 3)`. Split triangles into `A, B, C: (NT, 3)` each. Compute `e1 = B - A`, `e2 = C - A` — `(NT, 3)`.
2. **Repeat-broadcast** to the (NR, NT, 3) shape, all stride-0 views:
   - `O_b = repeat(O, 'r d -> r t d', t=NT)`
   - `D_b = repeat(D, 'r d -> r t d', t=NT)`
   - `A_b = repeat(A, 't d -> r t d', r=NR)`
   - `e1_b = repeat(e1, 't d -> r t d', r=NR)`, `e2_b = repeat(e2, 't d -> r t d', r=NR)`
3. Build LHS `(NR, NT, 3, 3)` by stacking `[-D_b, e1_b, e2_b]` as COLUMNS (stack along `dim=-1`).
4. Build RHS `(NR, NT, 3)` = `O_b - A_b`.
5. **linalg-solve-batched**: `t.linalg.solve(LHS, RHS)` — the leading `(NR, NT)` is the batch shape; LAPACK handles all NR*NT solves in one shot.

Return shape `(NR, NT, 3)`. Cross-check against the per-pair Python-loop reference (the test computes both and asserts allclose).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx11_intersect_all_pairs(rays, triangles):
    raise NotImplementedError

def _test_cx11():
    # atom-coverage: enforce that the solution actually uses einops.repeat
    # (not .expand() or t.broadcast_to or a full-copy tensor). Without this
    # check the value-tests pass with .expand and the einops-repeat atom
    # claim is fig-leaf.
    import inspect
    _src = inspect.getsource(cx11_intersect_all_pairs)
    assert 'repeat(' in _src, 'solution must use einops.repeat (not .expand/.broadcast_to/full-copy)'
    # Case A: small case — cross-check against a Python-loop reference using cx10's logic.
    t.manual_seed(0)
    NR, NT = 4, 3
    rays = t.randn(NR, 2, 3)
    # Force directions to point along -z so all solves are well-posed.
    rays[:, 1] = t.tensor([0.1, 0.2, -1.0])
    tri_centers = t.randn(NT, 3) * 0.1
    triangles = t.stack([
        tri_centers,
        tri_centers + t.tensor([1.0, 0.0, 0.0]),
        tri_centers + t.tensor([0.0, 1.0, 0.0]),
    ], dim=1)  # (NT, 3, 3)
    out = cx11_intersect_all_pairs(rays, triangles)
    assert tuple(out.shape) == (NR, NT, 3), f'shape: {tuple(out.shape)}'

    # Reference loop.
    ref = t.zeros(NR, NT, 3)
    for ri in range(NR):
        O = rays[ri, 0]; D = rays[ri, 1]
        for ti in range(NT):
            A = triangles[ti, 0]; B = triangles[ti, 1]; C = triangles[ti, 2]
            LHS = t.stack([-D, B - A, C - A], dim=-1)
            RHS = O - A
            ref[ri, ti] = t.linalg.solve(LHS, RHS)
    assert t.allclose(out, ref, atol=1e-4), f'max diff {(out - ref).abs().max()}'

    # Case B: reconstruction round-trip on one slot.
    ri, ti = 2, 1
    s, u, v = out[ri, ti]
    O = rays[ri, 0]; D = rays[ri, 1]
    A = triangles[ti, 0]; e1 = triangles[ti, 1] - A; e2 = triangles[ti, 2] - A
    lhs_pt = O + s * D
    rhs_pt = A + u * e1 + v * e2
    assert t.allclose(lhs_pt, rhs_pt, atol=1e-4), f'point mismatch: {lhs_pt} vs {rhs_pt}'

    # Case C: realistic ARENA-ish scale.
    rays3 = t.randn(30, 2, 3)
    rays3[:, 1] = t.tensor([0.0, 0.0, -1.0])
    tri3 = t.randn(20, 3, 3)
    out3 = cx11_intersect_all_pairs(rays3, tri3)
    assert tuple(out3.shape) == (30, 20, 3)
    _dd_passed.add('cx11')

_test_cx11()

<details><summary>Show solution — cx11</summary>

```python
def cx11_intersect_all_pairs(rays, triangles):
    NR = rays.shape[0]
    NT = triangles.shape[0]
    O = rays[:, 0]    # (NR, 3)
    D = rays[:, 1]    # (NR, 3)
    A = triangles[:, 0]  # (NT, 3)
    e1 = triangles[:, 1] - A  # (NT, 3)
    e2 = triangles[:, 2] - A  # (NT, 3)
    # Atom A (einops-repeat-broadcast): align everything to (NR, NT, 3) as stride-0 views.
    O_b  = repeat(O,  'r d -> r t d', t=NT)
    D_b  = repeat(D,  'r d -> r t d', t=NT)
    A_b  = repeat(A,  't d -> r t d', r=NR)
    e1_b = repeat(e1, 't d -> r t d', r=NR)
    e2_b = repeat(e2, 't d -> r t d', r=NR)
    # Stack columns [-D, e1, e2] to form a (NR, NT, 3, 3) LHS.
    LHS = t.stack([-D_b, e1_b, e2_b], dim=-1)
    RHS = O_b - A_b
    # Atom B (linalg-solve-batched): the leading (NR, NT) is the batch shape.
    return t.linalg.solve(LHS, RHS)
```

`torch.linalg.solve` accepts arbitrary batch shapes — `(NR, NT, 3, 3)` is fine, the function broadcasts/loops internally. Because all the einops repeats are stride-0 views, the input footprint stays O(NR + NT), not O(NR*NT). The fused LAPACK call is the only meaningful allocation. This is exactly the form ARENA uses for its `raytrace_mesh` kernel.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx11'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx11',
        'subtopics': ["Einops: Repeat-as-broadcast", "PyTorch: Batched linalg.solve"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()